In [ ]:
import sys
from pathlib import Path

# Add skill-test to path
repo_root = Path(".").resolve()
while not (repo_root / ".test" / "src").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / ".test" / "src"))

print(f"Repo root: {repo_root}")

## Step 1: Inspect the Skill

Let's look at the `databricks-model-serving` skill -- its current size, test cases, and baseline score.

In [ ]:
# Show first few test cases
for t in train[:3]:
    print(f"\n--- {t['id']} ---")
    print(f"Prompt: {t['input'][:100]}...")
    if t.get('answer'):
        print(f"Answer: {t['answer'][:100]}...")

In [ ]:
from skill_test.optimize.evaluator import create_skill_evaluator, SKILL_KEY
from skill_test.optimize.splitter import to_gepa_instances

evaluator = create_skill_evaluator(SKILL_NAME)
seed_candidate = {SKILL_KEY: original_content}

# Evaluate on all train tasks
gepa_instances = to_gepa_instances(train)

print(f"{'Task ID':<35} {'Score':>8}")
print("-" * 45)
for i, inst in enumerate(gepa_instances):
    score, side_info = evaluator(seed_candidate, inst)
    task_id = train[i]['id']
    status = 'PASS' if score >= 0.5 else 'FAIL'
    print(f"{task_id:<35} {score:>7.3f}  {status}")

# Quick baseline
scores = [evaluator(seed_candidate, inst)[0] for inst in gepa_instances]
baseline_score = sum(scores) / len(scores)
print(f"\nBaseline Score: {baseline_score:.3f}")
print(f"Token Count:    {original_tokens:,}")

In [ ]:
from skill_test.optimize.runner import optimize_skill

result = optimize_skill(
    skill_name=SKILL_NAME,
    mode="static",
    preset="quick",  # 15 iterations -- increase to "standard" (50) or "thorough" (150) for better results
)

print(f"Optimization complete!")
print(f"GEPA metric calls: {result.gepa_result.total_metric_calls}")
print(f"Candidates explored: {result.gepa_result.num_candidates}")

In [ ]:
print("=" * 60)
print(f"  OPTIMIZATION RESULTS: {SKILL_NAME}")
print("=" * 60)
print()

# Quality comparison
quality_delta = result.improvement
quality_pct = (quality_delta / result.original_score * 100) if result.original_score > 0 else 0
print(f"  Quality Score")
print(f"    Before:      {result.original_score:.3f}")
print(f"    After:       {result.optimized_score:.3f}")
print(f"    Delta:       {quality_delta:+.3f} ({quality_pct:+.1f}%)")
print()

# Token comparison  
token_delta = result.original_token_count - result.optimized_token_count
print(f"  Token Count")
print(f"    Before:      {result.original_token_count:,}")
print(f"    After:       {result.optimized_token_count:,}")
print(f"    Saved:       {token_delta:,} tokens ({result.token_reduction_pct:.1f}% reduction)")
print()

# Line count comparison
orig_lines = len(result.original_content.splitlines())
opt_lines = len(result.optimized_content.splitlines())
print(f"  Lines")
print(f"    Before:      {orig_lines}")
print(f"    After:       {opt_lines}")
print(f"    Saved:       {orig_lines - opt_lines} lines")
print()

# Validation scores
if result.val_scores:
    avg_val = sum(result.val_scores.values()) / len(result.val_scores)
    print(f"  Validation (held-out test cases)")
    for tid, score in result.val_scores.items():
        print(f"    {tid}: {score:.3f}")
    print(f"    Average: {avg_val:.3f}")

print()
print("=" * 60)

## Step 5: Review the Diff

Inspect what GEPA changed in the SKILL.md.

## Step 6: Apply (Optional)

If the results look good, apply the optimized SKILL.md. Uncomment the cell below to write it.

## Multi-Component Optimization: Skills + Tools

GEPA supports optimizing multiple text components simultaneously. You can optimize SKILL.md files **alongside** MCP tool descriptions in a single run.

GEPA's `RoundRobinReflectionComponentSelector` cycles through components one at a time, so each gets dedicated reflection and mutation.

In [ ]:
## Changing the Reflection Model

By default, GEPA uses `databricks/databricks-gpt-5-2` via Databricks Model Serving.
Override per-call or via environment variable:

```python
# Per-call
result = optimize_skill("my-skill", reflection_lm="openai/gpt-4o")

# Environment variable (persistent)
os.environ["GEPA_REFLECTION_LM"] = "databricks/databricks-gpt-5-2"
```

See README.md for full model configuration options.